# GPT-4o gene-description generation and BERT encoding

This notebook documents the GPT-to-BERT text feature workflow. It expects `OPENAI_API_KEY` to be provided as an environment variable. Do not hard-code API keys in this notebook. `OPENAI_BASE_URL` is optional and defaults to the official OpenAI endpoint.


In [ ]:
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
import os
from openai import OpenAI


In [ ]:
def revise_phrase(text):  
    text = text.replace("尾-cell", "beta-cell")  
    text = text.replace("β-cell", "beta-cell")  
    # 查找以"As of"开头的句子，并删除到逗号为止的部分  
    if "As of" in text:  
        # 找到"As of"开始到逗号结束的部分  
        start_index = text.find("As of")  
        end_index = text.find(",", start_index) + 1  # 逗号后的位置  
        # 删除这部分，并将逗号后的文本转为大写  
        text = text[:start_index] + text[end_index:].strip().capitalize()  
    return text
api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Please set OPENAI_API_KEY in the environment before running this notebook.')

base_url = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')
client = OpenAI(api_key=api_key, base_url=base_url)

def get_gene_desc(filename):
    df=pd.read_csv(filename, index_col=0)

    cnt = 0
    for gene in df.index:
        cnt += 1
        if not pd.isnull(df.loc[gene, 'gene_desc']): continue
        completion = client.chat.completions.create(
          model="gpt-4o",
          messages=[
            {"role": "user", "content": f'Is there a relationship between {gene} gene and type 2 diabetes? If it exists, please describe the possible relevant mechanisms in a short paragraph, without mentioning whether the gene can be druggable, and try to be scientifically rigorous;If it does not exist, please answer that it is currently unknown'}
          ]
        )
        df.loc[gene, 'gene_desc']=completion.choices[0].message.content
        
        print(completion.choices[0].message.content)

        if cnt % 50 == 0:
            df.reset_index().to_csv('neg_desc.csv', index=False)
            
    df.reset_index(inplace=True)
    df['gene_desc'] = df['gene_desc'].apply(revise_phrase)  
    df.to_csv(f'{filename[:-4]}_desc.csv', index=False)  

In [ ]:
pos_name='../gene_desc_feature_gpt4o/pos.csv'
# neg_10_name='../gene_desc_feature_gpt4o/neg_10.csv'
neg_name='../gene_desc_feature_gpt4o/pos.csv'
# get_gene_desc(pos_name)
get_gene_desc(neg_name)

In [ ]:
pos_desc='../gene_desc_feature_gpt4o/pos_desc.csv'
neg_10_desc='../gene_desc_feature_gpt4o/neg_10_desc.csv'

In [ ]:
neg_desc='../gene_desc_feature_gpt4o/neg_desc.csv'

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  
model = BertModel.from_pretrained('bert-base-uncased')

def get_gene_desc_vector(filename):
    df=pd.read_csv(filename)
    df_tmp=pd.read_csv(filename,usecols=[0],index_col=0)
    lt=[f"desc_vector_{i}" for i in range(768)]
    for col in lt:
        df_tmp[col]=0.0

    for index, row in df.iterrows():  
        gene_name = row['gene_name']  
        gene_desc = row['gene_desc']  
        inputs = tokenizer(gene_desc, return_tensors='pt')  
        outputs = model(**inputs)  

        last_hidden_states = outputs.last_hidden_state  
        cls_output = last_hidden_states.squeeze(0)[0].detach().numpy()  
        
        for i in range(768):
            df_tmp.loc[gene_name,f'desc_vector_{i}']=cls_output[i]
            
    df_tmp.reset_index(inplace=True)
    df_tmp.to_csv(f'{filename[:-4]}_vector.csv',index=False)    

In [ ]:
get_gene_desc_vector(neg_desc)

In [ ]:
df_tmp

In [ ]:
get_gene_desc_vector(pos_desc)
get_gene_desc_vector(neg_10_desc)

In [ ]:
pos_0 = pd.read_csv('../gene_feature/pos_omics.csv', index_col=0)  
pos_1 = pd.read_csv('../gene_desc_feature_gpt4o/pos_desc_vector.csv', index_col=0)  

neg_10_0 = pd.read_csv('../gene_feature/neg_omics_random_10.csv', index_col=0)  
neg_10_1 = pd.read_csv('../gene_desc_feature_gpt4o/neg_10_desc_vector.csv', index_col=0)  


pos = pd.concat([pos_0,pos_1], axis=1)  
neg10 = pd.concat([neg_10_0,neg_10_1], axis=1)  

pos.reset_index().to_csv('../test_data_gpt4o/pos_all.csv',index=False)
neg10.reset_index().to_csv('../test_data_gpt4o/neg_all_random_10.csv',index=False)


In [ ]:
pos.shape

In [ ]:
pos.shape